In [1]:
import re                               # for document class
import xml.etree.ElementTree as ET      # for parsing xml
import os
import pandas as pd
# text processor
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
# end text processor
# encoplot engine
import subprocess
from sentence_transformers import SentenceTransformer, util
import torch
# end encoplot engine
import shutil                             # copy dataset

In [2]:
from google.colab import drive

# mount the drive
drive.mount('/content/drive')

# define the paths for the dataset
BASE_PATH = "/content/drive/MyDrive/PAN11/external-detection-corpus"

SOURCE_PATH = os.path.join(BASE_PATH, "source-document")
SUSPICIOUS_PATH = os.path.join(BASE_PATH, "suspicious-document")

# define the paths for the encoplot code
ENCOPLOT_PATH = "/content/drive/MyDrive/utils/encoplot.c"
EXECUTABLE_PATH = "./encoplot_engine"

print(f"Lookin for data in: {BASE_PATH}")

Mounted at /content/drive
Lookin for data in: /content/drive/MyDrive/PAN11/external-detection-corpus


In [3]:
!gcc -O3 {ENCOPLOT_PATH} -o {EXECUTABLE_PATH}

## Segment

In [4]:
class Segment:
  def __init__(self, raw_text, start_index, length, clean_text=None):
    self.raw_text = raw_text
    self.start_index = start_index
    self.length = length
    self.clean_text = clean_text

    #self.embedding_vec = None
    #self.pos_tags = None
    #self.concept_ids = None

  def __eq__(self, other):
    s_start, s_end = self.start_index, self.start_index + self.length
    o_start, o_end = other.start_index, other.start_index + other.length

    MAX_DIF = 50
    return not (s_end + MAX_DIF < o_start or o_end + MAX_DIF < s_start)

  def __repr__(self):
    return "offset: " + str(self.start_index) + " length: " + str(self.length)

In [5]:
class PlagiarismFeature:
    def __init__(self, this_offset, this_length, source_reference,
                 source_offset, source_length, obfuscation):
        self.this_offset = this_offset
        self.this_length = this_length
        self.source_reference = source_reference
        self.source_offset = source_offset
        self.source_length = source_length
        self.obfuscation = obfuscation

    def get_source_id(self):
        return int(self.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))

    def __repr__(self):
        return (f"PlagiarismFeature("
                f"offset={self.this_offset}, "
                f"length={self.this_length}, "
                f"source={self.source_reference}, "
                f"obfuscation={self.obfuscation})")

DOCUMENT

part number = ((ID - 1) / 500) + 1

In [6]:
class Document:
  def __init__(self, doc_id, is_source=True):
    self.numeric_id = int(doc_id)
    self.is_source = is_source

    formatted_id = f"{self.numeric_id:05d}"
    prefix = "source-document" if is_source else "suspicious-document"

    part_number = ((self.numeric_id - 1) // 500) + 1
    part_folder = f"part{part_number}"
    base_folder = SOURCE_PATH if is_source else SUSPICIOUS_PATH

    self.doc_name = f"{prefix}{formatted_id}.txt"
    self.xml_name = f"{prefix}{formatted_id}.xml"

    self.file_path = os.path.join(base_folder, part_folder, self.doc_name)
    self.xml_path = os.path.join(base_folder, part_folder, self.xml_name)

    with open(self.file_path, "r", encoding='utf-8', errors='ignore') as f:
      self.text = f.read()

    self.metadata = self._parse_xml()
    self.language = self.metadata.get('lang', 'english')
    self.segments = []

  def _parse_xml(self):
    meta = {}
    self.plagiarism_features = []
    try:
        tree = ET.parse(self.xml_path)
        root = tree.getroot()

        for feature in root.findall('feature'):
            if feature.get('name') == 'about':
                meta['lang'] = feature.get('lang', 'en')

            if feature.get('name') == 'md5Hash':
                meta['md5'] = feature.get('value')

            if feature.get('name') == 'plagiarism':
                self.plagiarism_features.append(PlagiarismFeature(
                    this_offset=int(feature.get('this_offset')),
                    this_length=int(feature.get('this_length')),
                    source_reference=feature.get('source_reference'),
                    source_offset=int(feature.get('source_offset')),
                    source_length=int(feature.get('source_length')),
                    obfuscation=feature.get('obfuscation', 'none')
                ))
    except Exception as e:
        print(f"Error parsing XML: {e}")
    return meta

  def extract_segments(self, anchor_offsets, window_size=256):
    self.segments = []
    for offset in anchor_offsets:
      start = max(0, offset)

      if start >= len(self.text): continue

      while start > 0 and self.text[start-1] not in [' ', '\n', '\t']:
        start -= 1

      end = min(len(self.text), start + window_size)

      while end < len(self.text) and self.text[end] not in [' ', '\n', '.', '!', '?']:
        end += 1

      self.segments.append(Segment(self.text[start:end], start, end - start))

ENCOPLOT ENGINE

In [7]:
class EncoplotEngine:
  def __init__(self, executable_path="./encoplot_engine"):
    self.executable_path = executable_path

  def get_anchors(self, path_susp, path_src):
    result = subprocess.run([self.executable_path, path_susp, path_src],
                            capture_output=True, text=True)
    if result.returncode != 0: return []

    lines = result.stdout.strip().split('\n')

    return [list(map(int, line.split())) for line in lines if line]

SEMANTICANALYZER

In [8]:
class SemanticAnalyzer:
  def __init__(self, model_name='paraphrase-multilingual-mpnet-base-v2'):  # paraphrase-multilingual-MiniLM-L12-v2
    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    self.model = SentenceTransformer(model_name).to(self.device)

  def analyze_pairs(self, doc_susp, doc_src, threshold=0.6):
    texts_susp = [re.sub(r'\s+', ' ', s.raw_text).strip().lower() for s in doc_susp.segments]
    texts_src = [re.sub(r'\s+', ' ', s.raw_text).strip().lower() for s in doc_src.segments]

    embeddings_susp = self.model.encode(texts_susp, convert_to_tensor=True, show_progress_bar=True)
    embeddings_src = self.model.encode(texts_src, convert_to_tensor=True, show_progress_bar=True)

    cosine_scores = torch.nn.functional.cosine_similarity(embeddings_susp, embeddings_src)

    confirmed = []
    for i, score in enumerate(cosine_scores):
      if score >= threshold:
        confirmed.append({
            'score': score.item(),
            'susp_offset': doc_susp.segments[i].start_index,
            'susp_len': doc_susp.segments[i].length,
            'src_offset': doc_src.segments[i].start_index,
            'src_len': doc_src.segments[i].length,
            'text': texts_susp[i]
        })

    return confirmed

In [9]:
class EncoplotResult:
    def __init__(self, susp_id, src_id, anchors):
        self.susp_id = susp_id
        self.src_id = src_id
        self.anchors = anchors # list of [off_susp, off_src]
        self.anchor_count = len(anchors)
        # density score
        self.density_score = self._calculate_density()

    def _calculate_density(self):
        if self.anchor_count < 2: return 0
        offsets = [a[0] for a in self.anchors]
        span = max(offsets) - min(offsets)
        return self.anchor_count / (span / 1000) if span > 0 else 0

In [26]:
def cluster_results(hits, proximity=500):
    if not hits: return []

    hits.sort(key=lambda x: x['susp_offset'])

    merged = []
    current = hits[0]

    for i in range(1, len(hits)):
        nxt = hits[i]

        if nxt['susp_offset'] <= current['susp_offset'] + current['susp_len'] + proximity:
            new_end = max(current['susp_offset'] + current['susp_len'], nxt['susp_offset'] + nxt['susp_len'])
            current['susp_len'] = new_end - current['susp_offset']
            current['score'] = max(current['score'], nxt['score'])
        else:
            merged.append(current)
            current = nxt

    merged.append(current)
    return merged

def get_safe_fragment(text, offset, win=450):
        if not text:
            return "", 0, 0

        text_len = len(text)
        start = max(0, min(offset, text_len - 1))

        while start > 0:
            if text[start-1] in [' ', '\n', '\t']:
                break
            start -= 1

        end = min(text_len, start + win)

        while end < text_len:
            if text[end] in [' ', '\n', '.', '!', '?']:
                break
            end += 1

        fragment = text[start:end]
        return fragment, start, end - start

TEXT PROCESSOR

!pip install --force-reinstall --no-cache-dir nltk

- daca nu merge nltk

In [27]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:
class TextPreprocessor:
  def __init__(self, language='english'):
     self.lemmatizer = WordNetLemmatizer()
     try:
      self.stop_words = set(stopwords.words(language))
     except:
      self.stop_words = set(stopwords.words('english'))

  def process_document(self, doc):
    for seg in doc.segments:
      text = seg.raw_text.lower()
      text = re.sub(r"'s\b|'s", "", text)
      text = re.sub(r'[^a-z\s]', ' ', text)
      text = re.sub(r'\s+', ' ', text).strip()

      tokens = nltk.word_tokenize(text)

      cleaned_tokens = [
          self.lemmatizer.lemmatize(w)
          for w in tokens
          if w not in self.stop_words and len(w) > 2
          ]

      seg.clean_text = " ".join(cleaned_tokens)

      seg.pos_tags = nltk.pos_tag(nltk.word_tokenize(seg.raw_text))

    print(f"[OK] final preporcessing for {len(doc.segments)} segments.")

In [13]:
class ValidationMetrics:
    def __init__(self):
        self.recalls = []
        self.precisions = []
        self.granularities = []

    def calculate_metrics(self, ground_truth_features, predicted_segments):
        if not ground_truth_features:
            return {"precision": 0, "recall": 1, "granularity": 1, "f1": 0}

        # recall
        total_recall = 0
        for gt in ground_truth_features:
            covered_len = 0
            for pred in predicted_segments:
                intersect_start = max(gt.this_offset, pred.susp_off)
                intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)

                if intersect_end > intersect_start:
                    covered_len += (intersect_end - intersect_start)

            total_recall += (covered_len / gt.this_length)

        avg_recall = total_recall / len(ground_truth_features)

        # precision
        total_precision = 0
        if not predicted_segments:
            avg_precision = 0
        else:
            for pred in predicted_segments:
                covered_len = 0
                for gt in ground_truth_features:
                    intersect_start = max(gt.this_offset, pred.susp_off)
                    intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                    if intersect_end > intersect_start:
                        covered_len += (intersect_end - intersect_start)
                total_precision += (covered_len / pred.susp_len)
            avg_precision = total_precision / len(predicted_segments)

        # granularity
        avg_gran = 1.0

        # F-measure (plagdet score)
        f1 = 0
        if avg_precision + avg_recall > 0:
            f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall)

        return {
            "recall": round(avg_recall, 4),
            "precision": round(avg_precision, 4),
            "f1": round(f1, 4)
        }

In [14]:
def extract_sample(n_suspicious):
    suspicious_docs = []
    source_ids_needed = set()

    for i in range(1, n_suspicious + 1):
        try:
            doc = Document(i, is_source=False)
            suspicious_docs.append(doc)

            for pf in doc.plagiarism_features:
                src_id = int(pf.source_reference
                               .replace("source-document", "")
                               .replace(".txt", ""))
                source_ids_needed.add(src_id)

        except Exception as e:
            print(f"[!] Suspicious {i:05d} error: {e}")
            continue

    print(f"Loaded suspicious docs : {len(suspicious_docs)}")
    print(f"Unique sources : {len(source_ids_needed)}")

    return suspicious_docs, sorted(source_ids_needed)

In [15]:
# test 1
susp_id = "00175"
src_id = "03930"

doc_susp = Document(susp_id, is_source=False)
doc_src = Document(src_id, is_source=True)

print(f"Testing encoplot between docs:\n{doc_susp.file_path}\n{doc_src.file_path}")

Testing encoplot between docs:
/content/drive/MyDrive/PAN11/external-detection-corpus/suspicious-document/part1/suspicious-document00175.txt
/content/drive/MyDrive/PAN11/external-detection-corpus/source-document/part8/source-document03930.txt


In [16]:
encoplot = EncoplotEngine()
found_pairs = encoplot.anchors = encoplot.get_anchors(doc_susp.file_path, doc_src.file_path)
print(f"ENCOPLOT found {len(found_pairs)} n-grams matches.")

ENCOPLOT found 3046 n-grams matches.


In [17]:
if found_pairs:
    p_susp, p_src = found_pairs[0]

    doc_susp.extract_segments([p_susp])
    doc_src.extract_segments([p_src])

    seg_susp = doc_susp.segments[0]
    seg_src = doc_src.segments[0]

    preprocessor = TextPreprocessor(language=doc_susp.language)
    preprocessor.process_document(doc_susp)

    preprocessor_src = TextPreprocessor(language=doc_src.language)
    preprocessor_src.process_document(doc_src)

    print("----- PREPROCESSED SEGMENTS ------")
    print(f"SUSP Clean: {seg_susp.clean_text[:150]}...")
    print(f"SRC  Clean: {seg_src.clean_text[:150]}...")

    words_susp = set(seg_susp.clean_text.split())
    words_src = set(seg_src.clean_text.split())
    common = words_susp.intersection(words_src)

    print(f"\nCommon words in encoplot: {len(common)}")
    print(f"exemple: {list(common)[:10]}")

    if len(common) > 5:
        print("\n[YES] plagiarism detected!")
    else:
        print("\n[NO] they probably arrent plagiarised")

[OK] final preporcessing for 1 segments.
[OK] final preporcessing for 1 segments.
----- PREPROCESSED SEGMENTS ------
SUSP Clean: september think shut last missive informing equally settle coat intelligence miss wearisome though clip delightful commute magnificent dome domain...
SRC  Clean: believe closed last letter informing safely ensconced hair breadth escape wearisome though time delightful journey magnificent roof empire way hotel...

Common words in encoplot: 6
exemple: ['though', 'magnificent', 'informing', 'last', 'wearisome', 'delightful']

[YES] plagiarism detected!


In [18]:
from sentence_transformers import SentenceTransformer, util

class SemanticComparator:
    def __init__(self, model_name='paraphrase-multilingual-mpnet-base-v2'):
        print("Loading SBERT model...")
        self.model = SentenceTransformer(model_name)

    def calculate_similarity(self, segment_susp, segment_src):
        embedding1 = self.model.encode(segment_susp.clean_text, convert_to_tensor=True)
        embedding2 = self.model.encode(segment_src.clean_text, convert_to_tensor=True)

        cosine_score = util.cos_sim(embedding1, embedding2)

        return cosine_score.item()

In [19]:
susp_id = 175
source_ids = [3930, 7541, 10192, 7545]
#source_ids = [3930]
THRESHOLD = 0.50

encoplot = EncoplotEngine()
analyzer = SemanticAnalyzer()

doc_susp = Document(susp_id, is_source=False)
print(f"Analyzing document {doc_susp.doc_name}...\n")

for s_id in source_ids:
    print(f" - source {s_id:05d} ---")
    try:
        doc_src = Document(s_id, is_source=True)

        anchors = encoplot.get_anchors(doc_susp.file_path, doc_src.file_path)
        if not anchors:
            print("  [!] No anchors found.")
            continue

        texts_susp, texts_src, meta = [], [], []
        for p_susp, p_src in anchors:
            f_susp, s_off, s_len = get_safe_fragment(doc_susp.text, p_susp)
            f_src, src_off, src_len = get_safe_fragment(doc_src.text, p_src)

            t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
            t_src = re.sub(r'\s+', ' ', f_src).strip().lower()

            if len(t_susp) > 30:
                texts_susp.append(t_susp)
                texts_src.append(t_src)
                meta.append({'s_off': s_off, 's_len': s_len, 'src_off': src_off, 'src_len': src_len})

        if not texts_susp: continue

        emb_susp = analyzer.model.encode(texts_susp, convert_to_tensor=True, show_progress_bar=False)
        emb_src = analyzer.model.encode(texts_src, convert_to_tensor=True, show_progress_bar=False)
        scores = torch.nn.functional.cosine_similarity(emb_susp, emb_src)

        hits = []
        for i, score in enumerate(scores):
            if score >= THRESHOLD:
                m = meta[i]
                hits.append({
                    'score': score.item(),
                    'susp_offset': m['s_off'],
                    'susp_len': m['s_len'],
                    'src_offset': m['src_off'],
                    'src_len': m['src_len']
                })

        final_results = cluster_results(hits, proximity=1000)

        if final_results:
            print(f" [YES]: {len(final_results)} plagiarised fragments.")
            for r in final_results:
                print(f"     > Score: {r['score']:.2f} | Offset: {r['susp_offset']} | Length: {r['susp_len']}")
        else:
            print("  [NO] SBERT said ENCOPLOT is wrong :(")

    except Exception as e:
        print(f"Error in source {s_id}: {str(e)}")
    print("-" * 30)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Analyzing document suspicious-document00175.txt...

 - source 03930 ---
Error in source 3930: 'dict' object has no attribute 'susp_off'
------------------------------
 - source 07541 ---
Error in source 7541: 'dict' object has no attribute 'susp_off'
------------------------------
 - source 10192 ---
Error in source 10192: 'dict' object has no attribute 'susp_off'
------------------------------
 - source 07545 ---
  [NO] SBERT said ENCOPLOT is wrong :(
------------------------------


In [20]:
for s_id in source_ids:
    print(f"- comparing with source {s_id:05d} ---")
    try:
        doc_src = Document(s_id, is_source=True)
        anchors = encoplot.get_anchors(doc_susp.file_path, doc_src.file_path)

        if not anchors:
            print("  [!] No anchors found.")
            continue

        texts_susp, texts_src, meta = [], [], []

        for p_susp, p_src in anchors:
            if p_susp >= len(doc_susp.text) or p_src >= len(doc_src.text):
                continue

            f_susp, s_off, s_len = get_safe_fragment(doc_susp.text, p_susp)
            f_src, src_off, src_len = get_safe_fragment(doc_src.text, p_src)

            if not f_susp or not f_src: continue

            t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
            t_src = re.sub(r'\s+', ' ', f_src).strip().lower()

            if len(t_susp) > 30:
                texts_susp.append(t_susp)
                texts_src.append(t_src)
                meta.append({'s_off': s_off, 's_len': s_len, 'src_off': src_off, 'src_len': src_len})

        if not texts_susp:
            print("  [-] Anchors out of bounds.")
            continue

        emb_susp = analyzer.model.encode(texts_susp, convert_to_tensor=True, show_progress_bar=False)
        emb_src = analyzer.model.encode(texts_src, convert_to_tensor=True, show_progress_bar=False)
        scores = torch.nn.functional.cosine_similarity(emb_susp, emb_src)

        hits = []
        for i, score in enumerate(scores):
            if score >= 0.60:
                m = meta[i]
                hits.append({
                    'score': score.item(),
                    'susp_offset': m['s_off'], 'susp_len': m['s_len'],
                    'src_offset': m['src_off'], 'src_len': m['src_len']
                })

        final_results = cluster_results(hits, proximity=5000)

        if final_results:
            print(f" [YES]: {len(final_results)} PLAGIARISED FRAGMENTS.")
            for r in final_results:
                print(f"     > Score: {r['score']:.2f} | Offset: {r['susp_offset']} | Length: {r['susp_len']}")
        else:
            print("  [-] SBERT said NO.")

    except Exception as e:
        import traceback
        print(f" Error at source {s_id}: {str(e)}")
    print("-" * 30)

- comparing with source 03930 ---
 Error at source 3930: 'dict' object has no attribute 'susp_off'
------------------------------
- comparing with source 07541 ---
 Error at source 7541: 'dict' object has no attribute 'susp_off'
------------------------------
- comparing with source 10192 ---
 Error at source 10192: 'dict' object has no attribute 'susp_off'
------------------------------
- comparing with source 07545 ---
  [-] SBERT said NO.
------------------------------


In [21]:
suspicious_docs, source_ids = extract_sample(100)

Loaded suspicious docs : 100
Unique sources : 128


In [22]:
encoplot = EncoplotEngine()
analyzer = SemanticAnalyzer()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
import time
from collections import defaultdict

THRESHOLD = 0.65

def run_pipeline_on_sample(n_suspicious):
    print("-"*80)
    print(f"[STEP 1] extracting sample: {n_suspicious} suspicious files")
    print("-"*80)
    suspicious_docs, source_ids = extract_sample(n_suspicious)
    print(f"necessary sources: {len(source_ids)}\n")

    print("[STEP 2] Loading source docs")
    source_docs = []
    for s_id in source_ids:
        try:
            source_docs.append(Document(s_id, is_source=True))
        except Exception as e:
            print(f"  [!] Error in source {s_id}: {e}")
    print(f"sources loaded: {len(source_docs)}\n")

    print("[STEP 3] Encoplot ")
    t3 = time.time()

    all_texts_susp = []
    all_texts_src  = []
    all_meta       = []

    for doc_susp in suspicious_docs:
        for doc_src in source_docs:
            try:
                anchors = encoplot.get_anchors(doc_susp.file_path, doc_src.file_path)
                if not anchors:
                    continue

                MAX_ANCHORS_PER_PAIR = 300
                if len(anchors) > MAX_ANCHORS_PER_PAIR:
                    doc_len = len(doc_susp.text)
                    bucket_size = max(1, doc_len // MAX_ANCHORS_PER_PAIR)
                    buckets = {}
                    for p_susp, p_src in anchors:
                        bucket = p_susp // bucket_size
                        if bucket not in buckets:
                            buckets[bucket] = (p_susp, p_src)
                    anchors = list(buckets.values())

                for p_susp, p_src in anchors:
                    if p_susp >= len(doc_susp.text) or p_src >= len(doc_src.text):
                        continue

                    f_susp, s_off, s_len     = get_safe_fragment(doc_susp.text, p_susp)
                    f_src,  src_off, src_len = get_safe_fragment(doc_src.text, p_src)

                    if not f_susp or not f_src:
                        continue

                    t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
                    t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

                    if len(t_susp) > 30:
                        all_texts_susp.append(t_susp)
                        all_texts_src.append(t_src)
                        all_meta.append({
                            'susp_id':  doc_susp.numeric_id,
                            'src_id':   doc_src.numeric_id,
                            's_off':    s_off,
                            's_len':    s_len,
                            'src_off':  src_off,
                            'src_len':  src_len,
                        })

            except Exception as e:
                print(f"  [!] Error {doc_susp.numeric_id} vs {doc_src.numeric_id}: {e}")
                continue

    t3_elapsed = time.time() - t3
    print(f"Cndidate fragments: {len(all_texts_susp)} | Time: {t3_elapsed/60:.1f}min\n")

    all_texts_susp, all_texts_src, all_meta = prefilter_with_tfidf(all_texts_susp, all_texts_src, all_meta, tfidf_threshold=0.1)

    print(f"[STEP 4] SBERT batch encoding for {len(all_texts_susp)*2} suspicious docs")
    t4 = time.time()

    emb_susp = analyzer.model.encode(all_texts_susp, convert_to_tensor=True,
                                      show_progress_bar=True, batch_size=512)
    emb_src  = analyzer.model.encode(all_texts_src,  convert_to_tensor=True,
                                      show_progress_bar=True, batch_size=512)
    scores   = torch.nn.functional.cosine_similarity(emb_susp, emb_src)

    t4_elapsed = time.time() - t4
    print(f"Encoding done in {t4_elapsed:.1f}s\n")

    print("[STEP 5] filtering and clustering")

    hits_per_pair = defaultdict(list)
    for i, score in enumerate(scores):
        if score >= THRESHOLD:
            m = all_meta[i]
            hits_per_pair[(m['susp_id'], m['src_id'])].append({
                'score':       score.item(),
                'susp_offset': m['s_off'],
                'susp_len':    m['s_len'],
                'src_offset':  m['src_off'],
                'src_len':     m['src_len'],
            })

    detections = defaultdict(list)
    for (susp_id, src_id), hits in hits_per_pair.items():
        clustered = cluster_results(hits)
        for fragment in clustered:
            fragment['src_id'] = src_id
            detections[susp_id].append(fragment)

    print(f"Pairs with detected plagiarism: {len(hits_per_pair)}\n")

    print("[PASUL 6] generating XML output...")
    os.makedirs("/content/output", exist_ok=True)

    for doc_susp in suspicious_docs:
        susp_id = doc_susp.numeric_id
        root = ET.Element("document", reference=doc_susp.doc_name)

        for det in detections.get(susp_id, []):
            ET.SubElement(root, "feature",
                name="detected-plagiarism",
                this_offset=str(det['susp_offset']),
                this_length=str(det['susp_len']),
                source_reference=f"source-document{det['src_id']:05d}.txt",
                source_offset=str(det['src_offset']),
                source_length=str(det['src_len'])
            )

        tree = ET.ElementTree(root)
        out_path = f"/content/output/{doc_susp.doc_name.replace('.txt', '.xml')}"
        tree.write(out_path, encoding='utf-8', xml_declaration=True)

    print(f"XML saved in /content/output/\n")

    print("[STEP 7] detected vs ground truth...")
    print("-"*60)

    total_tp = total_fp = total_fn = 0

    for doc_susp in suspicious_docs:
        susp_id = doc_susp.numeric_id

        gt_fragments = [
            (pf.this_offset, pf.this_offset + pf.this_length)
            for pf in doc_susp.plagiarism_features
        ]

        detected_fragments = [
            (d['susp_offset'], d['susp_offset'] + d['susp_len'])
            for d in detections.get(susp_id, [])
        ]

        def covered_chars(fragments):
            covered = set()
            for start, end in fragments:
                covered.update(range(start, end))
            return covered

        gt_chars       = covered_chars(gt_fragments)
        detected_chars = covered_chars(detected_fragments)

        tp = len(gt_chars & detected_chars)
        fp = len(detected_chars - gt_chars)
        fn = len(gt_chars - detected_chars)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        if gt_fragments or detected_fragments:
            print(f"  Suspicios {susp_id:05d} | "
                  f"GT: {len(gt_fragments)} fragments | "
                  f"Detected: {len(detected_fragments)} | "
                  f"P: {precision:.2f} R: {recall:.2f} F1: {f1:.2f}")

    precision_g = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall_g    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1_g        = 2 * precision_g * recall_g / (precision_g + recall_g) if (precision_g + recall_g) > 0 else 0

    print("-"*60)
    print(f"SCORE")
    print(f"  Precision : {precision_g:.4f}")
    print(f"  Recall    : {recall_g:.4f}")
    print(f"  F1        : {f1_g:.4f}")
    print("-"*60)

    return suspicious_docs, source_docs, detections

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def prefilter_with_tfidf(texts_susp, texts_src, all_meta, tfidf_threshold=0.1):
    print(f"Prefilltering with tf-idf on {len(texts_susp)} pairs")

    all_texts = texts_susp + texts_src
    vectorizer = TfidfVectorizer(max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    n = len(texts_susp)
    susp_matrix = tfidf_matrix[:n]
    src_matrix  = tfidf_matrix[n:]

    filtered_susp, filtered_src, filtered_meta = [], [], []

    BATCH = 10000
    for i in range(0, n, BATCH):
        batch_susp = susp_matrix[i:i+BATCH]
        batch_src  = src_matrix[i:i+BATCH]

        scores = np.array(batch_susp.multiply(batch_src).sum(axis=1)).flatten()

        for j, score in enumerate(scores):
            if score >= tfidf_threshold:
                filtered_susp.append(texts_susp[i+j])
                filtered_src.append(texts_src[i+j])
                filtered_meta.append(all_meta[i+j])

    print(f"  after tf-idf: {len(filtered_susp)} remaining pairs "
          f"(crossed out: {n - len(filtered_susp)})")
    return filtered_susp, filtered_src, filtered_meta

In [28]:
suspicious_docs, source_docs, detections = run_pipeline_on_sample(n_suspicious=10)

--------------------------------------------------------------------------------
[STEP 1] extracting sample: 10 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 10
Unique sources : 3
necessary sources: 3

[STEP 2] Loading source docs
sources loaded: 3

[STEP 3] Encoplot 
Cndidate fragments: 3371 | Time: 0.0min

Prefilltering with tf-idf on 3371 pairs
  after tf-idf: 1500 remaining pairs (crossed out: 1871)
[STEP 4] SBERT batch encoding for 3000 suspicious docs


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Encoding done in 21.8s

[STEP 5] filtering and clustering
Pairs with detected plagiarism: 7

[PASUL 6] generating XML output...
XML saved in /content/output/

[STEP 7] detected vs ground truth...
------------------------------------------------------------
  Suspicios 00002 | GT: 0 fragments | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00
  Suspicios 00004 | GT: 0 fragments | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00
  Suspicios 00005 | GT: 1 fragments | Detected: 1 | P: 0.90 R: 0.99 F1: 0.94
  Suspicios 00006 | GT: 0 fragments | Detected: 2 | P: 0.00 R: 0.00 F1: 0.00
  Suspicios 00007 | GT: 6 fragments | Detected: 4 | P: 0.98 R: 0.75 F1: 0.85
  Suspicios 00008 | GT: 0 fragments | Detected: 1 | P: 0.00 R: 0.00 F1: 0.00
  Suspicios 00010 | GT: 6 fragments | Detected: 6 | P: 0.96 R: 0.78 F1: 0.86
------------------------------------------------------------
SCORE
  Precision : 0.8900
  Recall    : 0.7801
  F1        : 0.8315
------------------------------------------------------------


In [35]:
import nbformat

with open('/content/drive/MyDrive/Colab Notebooks/licenta.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

with open('/content/drive/MyDrive/Colab Notebooks/licenta_clean.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("Done!")

Done!
